# 06 — Executive Report
### Chapter 6 of 6 — Marketing + Finance Analytics Case Study

This closing chapter packages Chapters 1-5 into what would actually go in
front of leadership: a one-screen KPI dashboard, a plain-English executive
summary, and a prioritized, quantified set of strategic recommendations.
This notebook's figures and findings are the source material for
`reports/Executive_Report.pdf`.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import utils as u

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

u.apply_chart_theme()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

campaigns = pd.read_csv(u.DATA_RAW / "campaigns_clean.csv", parse_dates=["date"])
customers = pd.read_csv(u.DATA_RAW / "customers.csv", parse_dates=["acquisition_date"])
transactions = pd.read_csv(u.DATA_RAW / "transactions_clean.csv", parse_dates=["transaction_date"])
ab_test = pd.read_csv(u.DATA_RAW / "ab_test_campaigns.csv", parse_dates=["date"])

## Portfolio-Level KPI Scorecard

**Business Question:** In ten numbers, what is the state of the business?

**Methodology:** Recompute every core marketing and financial KPI at the
full-portfolio (blended) level, using the same formulas defined in
`src/utils.py` and applied per-channel in Chapters 2-4.

In [2]:
paid = campaigns[campaigns["spend"] > 0]
paid_txns = transactions[~transactions["refund_flag"]]

total_revenue = campaigns["revenue"].sum()
total_spend = campaigns["spend"].sum()
total_gross_profit = paid_txns["profit"].sum()
total_net_profit = total_gross_profit - total_spend

blended_cac = total_spend / len(customers)
blended_ltv = paid_txns.groupby("customer_id")["profit"].sum().mean()
ltv_cac_ratio = blended_ltv / blended_cac

blended_roas = u.roas(pd.Series([total_revenue]), pd.Series([total_spend]))[0]
blended_roi = u.roi(pd.Series([total_gross_profit]), pd.Series([total_spend]))[0]
blended_cvr = u.cvr(pd.Series([campaigns['conversions'].sum()]), pd.Series([campaigns['clicks'].sum()]))[0]

# Overall retention: share of customers with >1 non-refunded purchase (repeat-purchase rate)
purchase_counts = paid_txns.groupby("customer_id").size()
repeat_rate = (purchase_counts > 1).mean()

kpi_scorecard = {
    "Revenue": u.fmt_currency(total_revenue),
    "Marketing Spend": u.fmt_currency(total_spend),
    "Net Profit": u.fmt_currency(total_net_profit),
    "CAC (blended)": u.fmt_currency(blended_cac, 2),
    "LTV (blended)": u.fmt_currency(blended_ltv, 2),
    "LTV : CAC Ratio": f"{ltv_cac_ratio:.2f} : 1",
    "ROAS": f"{blended_roas:.2f}x",
    "ROI": u.fmt_pct(blended_roi),
    "Conversion Rate": u.fmt_pct(blended_cvr, 2),
    "Repeat Purchase Rate": u.fmt_pct(repeat_rate),
}
pd.Series(kpi_scorecard)

Revenue                 $1,291,170
Marketing Spend           $201,634
Net Profit              $1,134,157
CAC (blended)               $33.61
LTV (blended)              $223.71
LTV : CAC Ratio           6.66 : 1
ROAS                         6.40x
ROI                         562.5%
Conversion Rate              4.17%
Repeat Purchase Rate         83.7%
dtype: str

## Revenue Forecast (next quarter)

Reproduced from Chapter 5's best-performing backtested method (Linear
Trend), for inclusion on the dashboard.

In [3]:
monthly_revenue = campaigns.groupby(pd.Grouper(key="date", freq="MS"))["revenue"].sum()
monthly_revenue.index = monthly_revenue.index.to_period("M")

x_full = np.arange(len(monthly_revenue))
slope_f, intercept_f = np.polyfit(x_full, monthly_revenue.values, 1)
future_periods = pd.period_range(monthly_revenue.index[-1] + 1, periods=3, freq="M")
future_x = np.arange(len(monthly_revenue), len(monthly_revenue) + 3)
forecast_values = slope_f * future_x + intercept_f
next_quarter_forecast_total = forecast_values.sum()

print(f"Forecasted next-quarter revenue: {u.fmt_currency(next_quarter_forecast_total)}")

Forecasted next-quarter revenue: $64,346


## Interactive Executive KPI Dashboard

**Visualization:** A single-screen Plotly dashboard combining ten headline
KPI cards with the two charts leadership asks for first — the revenue
trend (with forecast) and channel-level ROAS/profit.

In [4]:
fig = make_subplots(
    rows=4, cols=5,
    specs=[
        [{"type": "indicator"}] * 5,
        [{"type": "indicator"}] * 5,
        [{"type": "xy", "colspan": 5}, None, None, None, None],
        [{"type": "xy", "colspan": 5}, None, None, None, None],
    ],
    row_heights=[0.16, 0.16, 0.34, 0.34],
    vertical_spacing=0.08,
    subplot_titles=("", "", "", "", "", "", "", "", "", "",
                     "Monthly Revenue: Historical + Next-Quarter Forecast",
                     "Net Profit by Channel"),
)

indicator_specs = [
    ("Revenue", total_revenue, "$,.0f", "", 1, 1),
    ("Marketing Spend", total_spend, "$,.0f", "", 1, 2),
    ("Net Profit", total_net_profit, "$,.0f", "", 1, 3),
    ("CAC (blended)", blended_cac, "$,.2f", "", 1, 4),
    ("LTV (blended)", blended_ltv, "$,.2f", "", 1, 5),
    ("LTV:CAC Ratio", ltv_cac_ratio, ",.2f", " : 1", 2, 1),
    ("ROAS", blended_roas, ",.2f", "x", 2, 2),
    ("ROI", blended_roi * 100, ",.1f", "%", 2, 3),
    ("Conversion Rate", blended_cvr * 100, ",.2f", "%", 2, 4),
    ("Repeat Purchase Rate", repeat_rate * 100, ",.1f", "%", 2, 5),
]
card_colors = (u.CATEGORICAL * 2)[:10]

for (label, value, fmt, suffix, r, c), color in zip(indicator_specs, card_colors):
    fig.add_trace(go.Indicator(
        mode="number",
        value=value,
        number={"valueformat": fmt, "suffix": suffix, "font": {"size": 28, "color": color}},
        title={"text": label, "font": {"size": 13, "color": u.CHROME['secondary_ink']}},
    ), row=r, col=c)

fig.add_trace(go.Scatter(x=monthly_revenue.index.to_timestamp(), y=monthly_revenue.values,
                          mode="lines", name="Historical Revenue",
                          line=dict(color=u.CATEGORICAL[0], width=2)), row=3, col=1)
fig.add_trace(go.Scatter(x=future_periods.to_timestamp(), y=forecast_values,
                          mode="lines+markers", name="Forecast",
                          line=dict(color=u.CATEGORICAL[2], width=2, dash="dash")), row=3, col=1)

channel_profit = (paid_txns.merge(customers[["customer_id", "acquisition_channel"]], on="customer_id")
                   .groupby("acquisition_channel")["profit"].sum()
                   - campaigns.groupby("channel")["spend"].sum()).sort_values()
fig.add_trace(go.Bar(x=channel_profit.values, y=channel_profit.index, orientation="h",
                      marker_color=[u.STATUS["good"] if v >= 0 else u.STATUS["critical"] for v in channel_profit.values],
                      showlegend=False), row=4, col=1)

fig.update_layout(
    height=1050, width=1150,
    title=dict(text="Executive KPI Dashboard — Marketing + Finance Performance (2024-2025)",
               font=dict(size=18), y=0.985),
    showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=0.615, x=0.01),
    margin=dict(t=110, l=60, r=30, b=40),
)
u.save_fig(fig, "06_executive_dashboard", is_plotly=True, width=1150, height=1000)
fig.show()

## Executive Summary

The business generated the revenue, spend, and profit shown on the
scorecard above across a seven-channel, five-country marketing program
over the trailing two years. Three findings dominate the picture:

1. **Revenue leadership and profit leadership are not the same channels.**
   Chapter 4 shows a measurable rank shift between top-line revenue and
   true net profit once marketing cost and category margins are applied —
   the single most consequential finding in this case study, because it
   means budget decisions made on a revenue dashboard alone would be wrong.
2. **Customer retention, not just acquisition, is a controllable profit
   lever.** The month-0-to-1 retention cliff (Chapter 3) and the
   channel-level LTV:CAC spread (Chapter 4) both point to the same
   conclusion: a dollar spent improving retention on already-acquired
   customers currently outperforms a dollar spent acquiring new ones on
   the weakest channels.
3. **We have a statistically validated, ready-to-execute creative win**
   (Chapter 5's Variant B) and a **quantified budget-reallocation plan**
   (Chapter 4) that can both be implemented immediately, with no new data
   collection required.

## Key Findings

- Net profit ranking diverges from revenue ranking at the channel level —
  see Chapter 4's `rank_shift` analysis.
- LTV:CAC ratio is below the 1:1 break-even line for at least one paid
  channel, meaning that channel is losing money per customer over their
  full lifetime despite reasonable first-touch ROAS.
- Retention decays most sharply in the first month post-acquisition, and
  varies meaningfully by acquisition channel (Chapter 3).
- Ad creative Variant B beats the control with p < 0.05 on two independent
  statistical tests (Chapter 5) — a genuine, not noise-driven, effect.
- A 15%-of-spend reallocation from the two weakest to two strongest
  elastic paid channels is projected to be profit-positive (Chapter 4).
- Next-quarter revenue is forecast using the best-backtested method, with
  its RMSE reported as the honest uncertainty band (Chapter 5).

## Strategic Recommendations

Each recommendation follows: **Finding -> Evidence -> Business Impact ->
Recommendation -> Expected Outcome**, and every expected outcome is
quantified using figures computed in Chapters 2-5 of this case study.

In [5]:
recommendations = pd.DataFrame([
    dict(
        finding="Two elastic paid channels (Instagram, TikTok) have the weakest ROI in the portfolio",
        evidence="Chapter 4 LTV:CAC and ROI ranking; both channels sit at/near break-even",
        business_impact="Marketing dollars are earning below-average return on these channels",
        recommendation="Shift 15% of spend from Instagram + TikTok into Google Ads + Affiliate, phased 50% test tranche then full rollout",
        expected_outcome="Chapter 4 model: net profit increase, computed from each channel's current ROI (see notebook 04 for the exact dollar figures)",
    ),
    dict(
        finding="Ad creative Variant B statistically outperforms the current control",
        evidence="Two-proportion z-test p=0.011; Welch's t-test p<0.0001; 95% CI excludes zero",
        business_impact="Continuing to run Variant A on any share of spend leaves conversions on the table",
        recommendation="Roll out Variant B to 100% of the Google Ads conversion campaign's budget",
        expected_outcome="Chapter 5 model: incremental conversions and revenue per 30-day period, computed from the observed lift",
    ),
    dict(
        finding="Month 0-to-1 customer retention is the steepest drop in the entire cohort curve",
        evidence="Chapter 3 cohort retention heatmap",
        business_impact="Every acquisition dollar loses a large share of its potential lifetime value in the first 30 days",
        recommendation="Launch an automated onboarding / second-purchase nudge triggered within 30 days of acquisition",
        expected_outcome="A retention lift here compounds through every later cohort month; even a modest improvement raises blended LTV and therefore LTV:CAC portfolio-wide",
    ),
    dict(
        finding="Email is the highest-ROAS, highest-LTV:CAC channel but is capacity-, not efficiency-, constrained",
        evidence="Chapter 2 & 4 channel KPI tables",
        business_impact="This channel is under-invested relative to its proven return",
        recommendation="Increase investment in list growth and content production for the Email program",
        expected_outcome="Additional volume at a return profile close to the current near-best-in-portfolio ROAS/LTV:CAC",
    ),
    dict(
        finding="India combines the lowest CPC with a competitive ROAS",
        evidence="Chapter 2 country performance comparison",
        business_impact="Currently under-scaled relative to its cost-efficiency",
        recommendation="Pilot a 20% incremental spend increase in India, monitoring ROAS as volume scales",
        expected_outcome="Tests whether the market's efficiency holds at scale before committing further budget",
    ),
])
recommendations

,finding,evidence,business_impact,recommendation,expected_outcome
0,"Two elastic paid channels (Instagram, TikTok) ...",Chapter 4 LTV:CAC and ROI ranking; both channe...,Marketing dollars are earning below-average re...,Shift 15% of spend from Instagram + TikTok int...,"Chapter 4 model: net profit increase, computed..."
1,Ad creative Variant B statistically outperform...,Two-proportion z-test p=0.011; Welch's t-test ...,Continuing to run Variant A on any share of sp...,Roll out Variant B to 100% of the Google Ads c...,Chapter 5 model: incremental conversions and r...
2,Month 0-to-1 customer retention is the steepes...,Chapter 3 cohort retention heatmap,Every acquisition dollar loses a large share o...,Launch an automated onboarding / second-purcha...,A retention lift here compounds through every ...
3,"Email is the highest-ROAS, highest-LTV:CAC cha...",Chapter 2 & 4 channel KPI tables,This channel is under-invested relative to its...,Increase investment in list growth and content...,Additional volume at a return profile close to...
4,India combines the lowest CPC with a competiti...,Chapter 2 country performance comparison,Currently under-scaled relative to its cost-ef...,Pilot a 20% incremental spend increase in Indi...,Tests whether the market's efficiency holds at...


## Estimated Financial Impact Summary

| Recommendation | Primary KPI Impact | Time Horizon |
|---|---|---|
| Budget reallocation (Instagram/TikTok -> Google Ads/Affiliate) | Net profit +, quantified in Chapter 4 | Immediate, phased over 2 months |
| Variant B rollout | Incremental conversions & revenue, quantified in Chapter 5 | Immediate |
| 30-day onboarding campaign | Blended LTV:CAC improvement (compounding) | 1-2 quarters to materialize |
| Email program investment | Incremental profit near current Email ROAS | 1 quarter |
| India spend pilot | Tests ROAS-at-scale; upside contingent on pilot result | 1 quarter (pilot), then scale decision |

**Bottom line:** Two of the five recommendations (budget reallocation and
Variant B rollout) can be executed immediately using only findings already
validated in this case study, with dollar-quantified expected impact
computed in Chapters 4 and 5. The remaining three (retention campaign,
Email investment, India pilot) require a short implementation cycle but
compound the return further.

## Conclusion

This case study demonstrates that a rigorous, six-chapter analytics
process — data validation, marketing efficiency, customer value,
financial truth, statistical and forecasting validation, and executive
synthesis — surfaces materially different (and more actionable)
conclusions than a single revenue dashboard would. The prioritized
recommendations above are ready for leadership review and, where
quantified, ready for immediate execution.

*End of case study. Full source data, SQL analysis, documentation, and
this report are available in the project repository — see `README.md`.*